# RetailMart Data Engineering Platform

## Problem Statement

RetailMart is a fast-growing e-commerce company that stores its operational data in multiple raw CSV files, including customer, order, product, payment, and order item information. Since these datasets are stored independently, the business team is unable to perform centralized reporting and answer important business questions such as monthly revenue, product performance, customer purchasing behavior, and payment trends.

The objective of this project is to build a centralized analytics platform that ingests raw data, performs data cleaning and transformation, maintains customer and product history, and generates business-ready datasets using the Medallion Architecture (Bronze, Silver, and Gold).

---

## Project Objectives

The following objectives are addressed in this project:

- Load raw CSV datasets into the analytics platform.
- Perform data quality checks and preprocessing.
- Build Bronze, Silver, and Gold data layers.
- Create a unified Customer 360 view.
- Maintain historical customer and product records using Slowly Changing Dimension (SCD Type 2).
- Generate business-ready analytical datasets.
- Answer business questions using Spark SQL.

---

## Dataset Description

The project uses the following datasets.

| Dataset | Description |
|----------|-------------|
| Customers | Customer master information |
| Orders | Customer order transactions |
| Products | Product catalogue |
| Order Items | Product details associated with each order |
| Payments | Payment information for each order |

---

## Technology Stack

| Component | Technology |
|-----------|------------|
| Programming Language | Python |
| Data Processing | Pandas, PySpark |
| Query Engine | Spark SQL |
| Storage Format | Parquet |
| Notebook Environment | Jupyter Notebook |
| Architecture | Medallion Architecture |

---

## Expected Output

The final solution provides a centralized analytics platform capable of supporting business reporting, customer analysis, product analytics, and revenue analysis using clean and standardized datasets.

# Dependency Management

This section verifies that all required Python libraries are available before executing the data engineering pipeline.

In [1]:
import importlib
import subprocess
import sys

required_packages = {
    "pandas": "pandas",
    "numpy": "numpy",
    "pyspark": "pyspark",
    "pyarrow": "pyarrow",
    "openpyxl": "openpyxl",
    "matplotlib": "matplotlib"
}

missing_packages = []

for package, module in required_packages.items():
    try:
        importlib.import_module(module)
        print(f"✓ {package}")
    except ImportError:
        print(f"✗ {package}")
        missing_packages.append(package)

if missing_packages:
    print("\nInstalling missing packages...\n")

    for package in missing_packages:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", package]
        )

    print("\nInstallation completed.")
else:
    print("\nAll required packages are available.")

✓ pandas
✓ numpy
✓ pyspark
✓ pyarrow
✓ openpyxl
✓ matplotlib

All required packages are available.


# Environment Setup

The required libraries are imported and an Apache Spark session is created. A local directory structure is also prepared for storing the Bronze, Silver, Gold, and Output datasets generated during the project.

In [16]:
import os
import warnings

import pandas as pd
import numpy as np

from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

warnings.filterwarnings("ignore")

In [17]:
spark = (
    SparkSession.builder
    .appName("RetailMart Data Engineering Platform")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

In [18]:
folders = [
    "data",
    "bronze",
    "silver",
    "gold",
    "output"
]

for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("Project folders created successfully.")

Project folders created successfully.


In [19]:
print("=" * 60)
print("Environment Information")
print("=" * 60)

print(f"Python      : {sys.version.split()[0]}")
print(f"Pandas      : {pd.__version__}")
print(f"NumPy       : {np.__version__}")
print(f"PySpark     : {spark.version}")

try:
    import pyarrow
    print(f"PyArrow     : {pyarrow.__version__}")
except:
    print("PyArrow     : Not Available")

try:
    import openpyxl
    print(f"OpenPyXL    : {openpyxl.__version__}")
except:
    print("OpenPyXL    : Not Available")

Environment Information
Python      : 3.14.5
Pandas      : 3.0.3
NumPy       : 2.4.6
PySpark     : 4.1.2
PyArrow     : 25.0.0
OpenPyXL    : 3.1.5


# 4. Data Ingestion

## Objective

The first step in the data engineering pipeline is to ingest the raw datasets into the processing environment. The RetailMart data is provided as separate CSV files representing customers, orders, products, order items, and payments.

For this project:

- Pandas is used for initial inspection and validation.
- PySpark is used for scalable data processing in the subsequent pipeline stages.

The datasets loaded in this section serve as the source for the Bronze, Silver, and Gold layers of the Medallion Architecture.

In [20]:
# ==========================================
# File Paths
# ==========================================

DATA_FOLDER = "data"

CUSTOMERS_FILE = os.path.join(DATA_FOLDER, "raw_customers_dataset.csv")
ORDERS_FILE = os.path.join(DATA_FOLDER, "raw_orders_dataset.csv")
PRODUCTS_FILE = os.path.join(DATA_FOLDER, "raw_products_dataset.csv")
ITEMS_FILE = os.path.join(DATA_FOLDER, "raw_items_dataset.csv")
PAYMENTS_FILE = os.path.join(DATA_FOLDER, "raw_payments_dataset.csv")

In [21]:
# ==========================================
# Load Raw Data using Pandas
# ==========================================

customers_pd = pd.read_csv(CUSTOMERS_FILE)
orders_pd = pd.read_csv(ORDERS_FILE)
products_pd = pd.read_csv(PRODUCTS_FILE)
items_pd = pd.read_csv(ITEMS_FILE)
payments_pd = pd.read_csv(PAYMENTS_FILE)

print("All datasets loaded successfully.")

All datasets loaded successfully.


In [22]:
# ==========================================
# Dataset Summary
# ==========================================

datasets = {
    "Customers": customers_pd,
    "Orders": orders_pd,
    "Products": products_pd,
    "Order Items": items_pd,
    "Payments": payments_pd
}

summary = pd.DataFrame({
    "Dataset": datasets.keys(),
    "Rows": [df.shape[0] for df in datasets.values()],
    "Columns": [df.shape[1] for df in datasets.values()]
})

summary

,Dataset,Rows,Columns
0,Customers,15000,5
1,Orders,50000,8
2,Products,3000,9
3,Order Items,86328,7
4,Payments,57388,5


In [23]:
# ==========================================
# Preview First Five Records
# ==========================================

for name, df in datasets.items():
    print(f"\n{name}")
    display(df.head())


Customers


,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,CUST_000001,UNIQ_000001,13278,salvador,BA
1,CUST_000002,UNIQ_000002,42098,porto alegre,RS
2,CUST_000003,UNIQ_000003,28289,recife,PE
3,CUST_000004,UNIQ_000004,98696,salvador,BA
4,CUST_000005,UNIQ_000005,21395,maceio,AL



Orders


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,ORD_0000001,CUST_006571,delivered,2023-06-15 14:30:00,2023-06-16 12:30:00,2023-06-17 12:30:00,2023-06-19 14:30:00,2023-06-29 14:30:00
1,ORD_0000002,CUST_006956,cancelled,2021-06-12 11:02:00,2021-06-12 23:02:00,2021-06-13 23:02:00,NaN,2021-06-21 11:02:00
2,ORD_0000003,CUST_008373,delivered,2022-03-31 19:38:00,2022-04-01 04:38:00,2022-04-04 04:38:00,2022-04-14 19:38:00,2022-04-17 19:38:00
3,ORD_0000004,CUST_007986,delivered,2021-09-09 22:29:00,2021-09-10 11:29:00,2021-09-13 11:29:00,2021-09-23 22:29:00,2021-09-27 22:29:00
4,ORD_0000005,CUST_002810,shipped,2022-08-27 07:46:00,2022-08-28 03:46:00,2022-08-29 03:46:00,NaN,2022-09-15 07:46:00



Products


,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,PROD_000001,music,66,512,5,18964,11,25,52
1,PROD_000002,food,40,452,4,25798,57,42,49
2,PROD_000003,office,69,121,5,24989,45,16,10
3,PROD_000004,music,46,717,4,18661,91,8,48
4,PROD_000005,toys,29,632,5,26347,79,32,11



Order Items


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,ORD_0000001,1,PROD_001950,SELL_0224,2023-06-18 14:30:00,1754.70,20.17
1,ORD_0000002,1,PROD_000989,SELL_0405,2021-06-14 11:02:00,2105.20,45.48
2,ORD_0000003,1,PROD_000254,SELL_0283,2022-04-03 19:38:00,1627.94,78.29
3,ORD_0000004,1,PROD_001327,SELL_0437,2021-09-12 22:29:00,65.22,31.08
4,ORD_0000004,2,PROD_002714,SELL_0396,2021-09-12 22:29:00,946.39,44.43



Payments


,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,ORD_0000001,1,voucher,12,2775.42
1,ORD_0000001,2,credit_card,1,32.43
2,ORD_0000002,1,voucher,1,805.73
3,ORD_0000003,1,credit_card,6,508.88
4,ORD_0000004,1,boleto,2,1589.64


In [25]:
# ==========================================
# Load CSV Files using PySpark
# ==========================================

import os

DATA_FOLDER = "data"

customers_spark = spark.read.csv(
    os.path.join(DATA_FOLDER, "raw_customers_dataset.csv"),
    header=True,
    inferSchema=True
)

orders_spark = spark.read.csv(
    os.path.join(DATA_FOLDER, "raw_orders_dataset.csv"),
    header=True,
    inferSchema=True
)

products_spark = spark.read.csv(
    os.path.join(DATA_FOLDER, "raw_products_dataset.csv"),
    header=True,
    inferSchema=True
)

items_spark = spark.read.csv(
    os.path.join(DATA_FOLDER, "raw_items_dataset.csv"),
    header=True,
    inferSchema=True
)

payments_spark = spark.read.csv(
    os.path.join(DATA_FOLDER, "raw_payments_dataset.csv"),
    header=True,
    inferSchema=True
)

print("✓ All Spark DataFrames loaded successfully.")

✓ All Spark DataFrames loaded successfully.


# 5. Data Understanding

## Objective

Before transforming the raw datasets, it is important to understand their structure and assess their overall quality.

This section examines each dataset to identify potential data quality issues that may affect downstream processing. The assessment includes:

- Dataset dimensions
- Column names and data types
- Missing values
- Duplicate records
- Descriptive statistics
- Sample records

The observations from this analysis will guide the data cleaning and standardization performed in the Silver layer.

In [26]:
# ==========================================
# Dataset Dimensions
# ==========================================

for name, df in datasets.items():
    print(f"{name:15} : {df.shape[0]:6} Rows × {df.shape[1]} Columns")

Customers       :  15000 Rows × 5 Columns
Orders          :  50000 Rows × 8 Columns
Products        :   3000 Rows × 9 Columns
Order Items     :  86328 Rows × 7 Columns
Payments        :  57388 Rows × 5 Columns


In [27]:
# ==========================================
# Dataset Information
# ==========================================

for name, df in datasets.items():
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    display(df.info())


Customers
<class 'pandas.DataFrame'>
RangeIndex: 15000 entries, 0 to 14999
Data columns (total 5 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   customer_id               15000 non-null  str  
 1   customer_unique_id        15000 non-null  str  
 2   customer_zip_code_prefix  15000 non-null  int64
 3   customer_city             15000 non-null  str  
 4   customer_state            15000 non-null  str  
dtypes: int64(1), str(4)
memory usage: 1.0 MB


None


Orders
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       50000 non-null  str  
 1   customer_id                    50000 non-null  str  
 2   order_status                   50000 non-null  str  
 3   order_purchase_timestamp       50000 non-null  str  
 4   order_approved_at              50000 non-null  str  
 5   order_delivered_carrier_date   50000 non-null  str  
 6   order_delivered_customer_date  24966 non-null  str  
 7   order_estimated_delivery_date  50000 non-null  str  
dtypes: str(8)
memory usage: 8.6 MB


None


Products
<class 'pandas.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   product_id                  3000 non-null   str  
 1   product_category_name       3000 non-null   str  
 2   product_name_length         3000 non-null   int64
 3   product_description_length  3000 non-null   int64
 4   product_photos_qty          3000 non-null   int64
 5   product_weight_g            3000 non-null   int64
 6   product_length_cm           3000 non-null   int64
 7   product_height_cm           3000 non-null   int64
 8   product_width_cm            3000 non-null   int64
dtypes: int64(7), str(2)
memory usage: 264.5 KB


None


Order Items
<class 'pandas.DataFrame'>
RangeIndex: 86328 entries, 0 to 86327
Data columns (total 7 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   order_id             86328 non-null  str    
 1   order_item_id        86328 non-null  int64  
 2   product_id           86328 non-null  str    
 3   seller_id            86328 non-null  str    
 4   shipping_limit_date  86328 non-null  str    
 5   price                86328 non-null  float64
 6   freight_value        86328 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 8.7 MB


None


Payments
<class 'pandas.DataFrame'>
RangeIndex: 57388 entries, 0 to 57387
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   order_id              57388 non-null  str    
 1   payment_sequential    57388 non-null  int64  
 2   payment_type          57388 non-null  str    
 3   payment_installments  57388 non-null  int64  
 4   payment_value         57388 non-null  float64
dtypes: float64(1), int64(2), str(2)
memory usage: 3.3 MB


None

In [28]:
# ==========================================
# Missing Values
# ==========================================

for name, df in datasets.items():

    missing = (
        df.isnull()
          .sum()
          .reset_index()
    )

    missing.columns = ["Column", "Missing Values"]
    missing["Percentage"] = (
        missing["Missing Values"] / len(df) * 100
    ).round(2)

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    display(missing)


Customers


,Column,Missing Values,Percentage
0,customer_id,0,0.0
1,customer_unique_id,0,0.0
2,customer_zip_code_prefix,0,0.0
3,customer_city,0,0.0
4,customer_state,0,0.0



Orders


,Column,Missing Values,Percentage
0,order_id,0,0.00
1,customer_id,0,0.00
2,order_status,0,0.00
3,order_purchase_timestamp,0,0.00
4,order_approved_at,0,0.00
5,order_delivered_carrier_date,0,0.00
6,order_delivered_customer_date,25034,50.07
7,order_estimated_delivery_date,0,0.00



Products


,Column,Missing Values,Percentage
0,product_id,0,0.0
1,product_category_name,0,0.0
2,product_name_length,0,0.0
3,product_description_length,0,0.0
4,product_photos_qty,0,0.0
5,product_weight_g,0,0.0
6,product_length_cm,0,0.0
7,product_height_cm,0,0.0
8,product_width_cm,0,0.0



Order Items


,Column,Missing Values,Percentage
0,order_id,0,0.0
1,order_item_id,0,0.0
2,product_id,0,0.0
3,seller_id,0,0.0
4,shipping_limit_date,0,0.0
5,price,0,0.0
6,freight_value,0,0.0



Payments


,Column,Missing Values,Percentage
0,order_id,0,0.0
1,payment_sequential,0,0.0
2,payment_type,0,0.0
3,payment_installments,0,0.0
4,payment_value,0,0.0


In [29]:
# ==========================================
# Duplicate Records
# ==========================================

duplicate_summary = []

for name, df in datasets.items():

    duplicate_summary.append({
        "Dataset": name,
        "Duplicate Rows": df.duplicated().sum()
    })

duplicate_summary = pd.DataFrame(duplicate_summary)

duplicate_summary

,Dataset,Duplicate Rows
0,Customers,0
1,Orders,0
2,Products,0
3,Order Items,0
4,Payments,0


In [30]:
# ==========================================
# Data Types
# ==========================================

for name, df in datasets.items():

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    display(
        pd.DataFrame({
            "Column": df.columns,
            "Data Type": df.dtypes.astype(str)
        })
    )


Customers


,Column,Data Type
customer_id,customer_id,str
customer_unique_id,customer_unique_id,str
customer_zip_code_prefix,customer_zip_code_prefix,int64
customer_city,customer_city,str
customer_state,customer_state,str



Orders


,Column,Data Type
order_id,order_id,str
customer_id,customer_id,str
order_status,order_status,str
order_purchase_timestamp,order_purchase_timestamp,str
order_approved_at,order_approved_at,str
order_delivered_carrier_date,order_delivered_carrier_date,str
order_delivered_customer_date,order_delivered_customer_date,str
order_estimated_delivery_date,order_estimated_delivery_date,str



Products


,Column,Data Type
product_id,product_id,str
product_category_name,product_category_name,str
product_name_length,product_name_length,int64
product_description_length,product_description_length,int64
product_photos_qty,product_photos_qty,int64
product_weight_g,product_weight_g,int64
product_length_cm,product_length_cm,int64
product_height_cm,product_height_cm,int64
product_width_cm,product_width_cm,int64



Order Items


,Column,Data Type
order_id,order_id,str
order_item_id,order_item_id,int64
product_id,product_id,str
seller_id,seller_id,str
shipping_limit_date,shipping_limit_date,str
price,price,float64
freight_value,freight_value,float64



Payments


,Column,Data Type
order_id,order_id,str
payment_sequential,payment_sequential,int64
payment_type,payment_type,str
payment_installments,payment_installments,int64
payment_value,payment_value,float64


In [31]:
# ==========================================
# Descriptive Statistics
# ==========================================

for name, df in datasets.items():

    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)

    display(df.describe(include="all").T)


Customers


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
customer_id,15000,15000,CUST_000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_unique_id,15000,15000,UNIQ_000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_zip_code_prefix,15000.0,NaN,NaN,NaN,54722.2996,26037.49203,10004.0,31931.0,54512.0,77155.75,99996.0
customer_city,15000,20,fortaleza,831,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_state,15000,20,CE,831,NaN,NaN,NaN,NaN,NaN,NaN,NaN



Orders


,count,unique,top,freq
order_id,50000,50000,ORD_0000001,1
customer_id,50000,14481,CUST_012016,13
order_status,50000,5,delivered,24966
order_purchase_timestamp,50000,49065,2022-04-04 11:37:00,3
order_approved_at,50000,49080,2021-05-23 21:11:00,3
order_delivered_carrier_date,50000,49077,2021-11-22 02:29:00,4
order_delivered_customer_date,24966,24720,2021-01-27 12:43:00,3
order_estimated_delivery_date,50000,49069,2021-07-29 22:14:00,3



Products


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
product_id,3000,3000,PROD_000001,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_category_name,3000,15,garden,233,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_name_length,3000.0,NaN,NaN,NaN,44.942333,14.757132,20.0,33.0,45.0,58.0,70.0
product_description_length,3000.0,NaN,NaN,NaN,554.860667,259.245576,100.0,331.75,560.0,781.0,1000.0
product_photos_qty,3000.0,NaN,NaN,NaN,3.475667,1.696772,1.0,2.0,3.5,5.0,6.0
product_weight_g,3000.0,NaN,NaN,NaN,15257.391,8603.978215,122.0,7694.75,15590.0,22682.25,29996.0
product_length_cm,3000.0,NaN,NaN,NaN,55.426,26.031335,10.0,33.0,56.0,78.0,100.0
product_height_cm,3000.0,NaN,NaN,NaN,27.579333,13.180552,5.0,16.0,27.0,39.0,50.0
product_width_cm,3000.0,NaN,NaN,NaN,44.700333,20.397105,10.0,27.0,45.0,62.0,80.0



Order Items


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
order_id,86328,50000,ORD_0000025,5,NaN,NaN,NaN,NaN,NaN,NaN,NaN
order_item_id,86328.0,NaN,NaN,NaN,1.706457,1.001408,1.0,1.0,1.0,2.0,5.0
product_id,86328,3000,PROD_001281,51,NaN,NaN,NaN,NaN,NaN,NaN,NaN
seller_id,86328,500,SELL_0192,210,NaN,NaN,NaN,NaN,NaN,NaN,NaN
shipping_limit_date,86328,67699,2021-02-17 14:05:00,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN
price,86328.0,NaN,NaN,NaN,1258.18473,718.160922,15.02,635.71,1258.665,1879.92,2499.96
freight_value,86328.0,NaN,NaN,NaN,42.566145,21.663455,5.0,23.92,42.59,61.3725,80.0



Payments


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
order_id,57388,50000,ORD_0000001,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
payment_sequential,57388.0,NaN,NaN,NaN,1.128738,0.334912,1.0,1.0,1.0,1.0,2.0
payment_type,57388,4,credit_card,28563,NaN,NaN,NaN,NaN,NaN,NaN,NaN
payment_installments,57388.0,NaN,NaN,NaN,3.723008,3.775384,1.0,1.0,2.0,6.0,12.0
payment_value,57388.0,NaN,NaN,NaN,1325.128164,937.133008,10.0,442.6925,1289.895,2144.2025,2999.92


In [33]:
# ==========================================
# Primary Key Validation
# ==========================================

print("=" * 70)
print("Primary Key Validation")
print("=" * 70)

# Customers
print(f"Customers  : Duplicate customer_id      = {customers_pd['customer_id'].duplicated().sum()}")

# Orders
print(f"Orders     : Duplicate order_id         = {orders_pd['order_id'].duplicated().sum()}")

# Products
print(f"Products   : Duplicate product_id       = {products_pd['product_id'].duplicated().sum()}")

# Order Items
print(f"Items      : Duplicate (order_id, order_item_id) = {items_pd.duplicated(subset=['order_id', 'order_item_id']).sum()}")

# Payments
print(f"Payments   : Duplicate (order_id, payment_sequential) = {payments_pd.duplicated(subset=['order_id', 'payment_sequential']).sum()}")

Primary Key Validation
Customers  : Duplicate customer_id      = 0
Orders     : Duplicate order_id         = 0
Products   : Duplicate product_id       = 0
Items      : Duplicate (order_id, order_item_id) = 0
Payments   : Duplicate (order_id, payment_sequential) = 0


# 6. Bronze Layer

## Objective

The Bronze layer is the entry point of the Medallion Architecture. It stores the raw datasets exactly as they are received from the source system without applying any transformations.

The purpose of this layer is to:

- Preserve the original data.
- Create a reliable source for downstream processing.
- Enable data traceability and reproducibility.
- Separate raw ingestion from data cleaning.

In this project, each raw CSV dataset is converted into Parquet format and stored in the Bronze layer.

In [35]:
# ==========================================
# Bronze Layer Storage Paths
# ==========================================

BRONZE_PATH = "bronze"

bronze_tables = {
    "customers": f"{BRONZE_PATH}/customers",
    "orders": f"{BRONZE_PATH}/orders",
    "products": f"{BRONZE_PATH}/products",
    "items": f"{BRONZE_PATH}/items",
    "payments": f"{BRONZE_PATH}/payments"
}

In [36]:
# ==========================================
# Write Raw Data to Bronze Layer
# ==========================================

customers_spark.write.mode("overwrite").parquet(bronze_tables["customers"])

orders_spark.write.mode("overwrite").parquet(bronze_tables["orders"])

products_spark.write.mode("overwrite").parquet(bronze_tables["products"])

items_spark.write.mode("overwrite").parquet(bronze_tables["items"])

payments_spark.write.mode("overwrite").parquet(bronze_tables["payments"])

print("Bronze layer created successfully.")

Bronze layer created successfully.


In [37]:
# ==========================================
# Read Bronze Layer
# ==========================================

bronze_customers = spark.read.parquet(bronze_tables["customers"])

bronze_orders = spark.read.parquet(bronze_tables["orders"])

bronze_products = spark.read.parquet(bronze_tables["products"])

bronze_items = spark.read.parquet(bronze_tables["items"])

bronze_payments = spark.read.parquet(bronze_tables["payments"])

In [46]:
# ==========================================
# Validate Bronze Layer
# ==========================================

print("=" * 60)
print("Bronze Layer Validation")
print("=" * 60)

print(f"{'Customers':<15}: {bronze_customers.count():>10,} rows")
print(f"{'Orders':<15}: {bronze_orders.count():>10,} rows")
print(f"{'Products':<15}: {bronze_products.count():>10,} rows")
print(f"{'Order Items':<15}: {bronze_items.count():>10,} rows")
print(f"{'Payments':<15}: {bronze_payments.count():>10,} rows")

print("\nBronze layer validation completed successfully.")

Bronze Layer Validation
Customers      :     15,000 rows
Orders         :     50,000 rows
Products       :      3,000 rows
Order Items    :     86,328 rows
Payments       :     57,388 rows

Bronze layer validation completed successfully.


# 7. Silver Layer

## Objective

The Silver layer transforms the raw Bronze datasets into clean and standardized datasets suitable for analytical processing.

The transformations performed in this layer include:

- Removing duplicate records
- Handling missing values
- Converting columns to appropriate data types
- Standardizing column names
- Validating numerical values
- Preparing datasets for business analytics

Unlike the Bronze layer, the Silver layer contains trusted and consistent data that will be used to build Gold layer reporting tables.

In [49]:
# ==========================================
# Silver Layer Storage
# ==========================================

SILVER_PATH = "silver"

silver_tables = {
    "customers": f"{SILVER_PATH}/customers",
    "orders": f"{SILVER_PATH}/orders",
    "products": f"{SILVER_PATH}/products",
    "items": f"{SILVER_PATH}/items",
    "payments": f"{SILVER_PATH}/payments"
}

In [50]:
# ==========================================
# Customers
# ==========================================

silver_customers = (
    bronze_customers
    .dropDuplicates(["customer_id"])
    .dropna(subset=["customer_id"])
)

print("Customer records :", silver_customers.count())

Customer records : 15000


In [51]:
# ==========================================
# Orders
# ==========================================

from pyspark.sql.functions import col,to_timestamp

silver_orders = (

    bronze_orders

    .dropDuplicates(["order_id"])

    .dropna(subset=["order_id","customer_id"])

    .withColumn(
        "order_purchase_timestamp",
        to_timestamp(col("order_purchase_timestamp"))
    )

    .withColumn(
        "order_approved_at",
        to_timestamp(col("order_approved_at"))
    )

    .withColumn(
        "order_delivered_carrier_date",
        to_timestamp(col("order_delivered_carrier_date"))
    )

    .withColumn(
        "order_delivered_customer_date",
        to_timestamp(col("order_delivered_customer_date"))
    )

    .withColumn(
        "order_estimated_delivery_date",
        to_timestamp(col("order_estimated_delivery_date"))
    )

)

print("Order records :", silver_orders.count())

Order records : 50000


In [52]:
# ==========================================
# Products
# ==========================================

silver_products = (

    bronze_products

    .dropDuplicates(["product_id"])

    .fillna({
        "product_category_name":"Unknown"
    })

)

print("Product records :", silver_products.count())

Product records : 3000


In [53]:
# ==========================================
# Order Items
# ==========================================

silver_items = (

    bronze_items

    .dropDuplicates(
        ["order_id","order_item_id"]
    )

    .filter(col("price")>=0)

    .filter(col("freight_value")>=0)

)

print("Item records :", silver_items.count())

Item records : 86328


In [54]:
# ==========================================
# Payments
# ==========================================

silver_payments = (

    bronze_payments

    .dropDuplicates(
        ["order_id","payment_sequential"]
    )

    .filter(col("payment_value")>=0)

)

print("Payment records :", silver_payments.count())

Payment records : 57388


In [55]:
# ==========================================
# Save Silver Layer
# ==========================================

silver_customers.write.mode("overwrite").parquet(
    silver_tables["customers"]
)

silver_orders.write.mode("overwrite").parquet(
    silver_tables["orders"]
)

silver_products.write.mode("overwrite").parquet(
    silver_tables["products"]
)

silver_items.write.mode("overwrite").parquet(
    silver_tables["items"]
)

silver_payments.write.mode("overwrite").parquet(
    silver_tables["payments"]
)

print("Silver layer created successfully.")

Silver layer created successfully.


In [56]:
# ==========================================
# Validate Silver Layer
# ==========================================

print("="*60)
print("Silver Layer Validation")
print("="*60)

print(f"Customers   : {silver_customers.count():>10,}")
print(f"Orders      : {silver_orders.count():>10,}")
print(f"Products    : {silver_products.count():>10,}")
print(f"Items       : {silver_items.count():>10,}")
print(f"Payments    : {silver_payments.count():>10,}")

Silver Layer Validation
Customers   :     15,000
Orders      :     50,000
Products    :      3,000
Items       :     86,328
Payments    :     57,388


## Summary

The datasets have been cleaned and standardized in the Silver layer. Duplicate records and invalid values were removed where applicable, missing values were handled, and date columns were converted to appropriate timestamp formats.

The resulting datasets provide a consistent and reliable foundation for creating business-ready analytical tables in the Gold layer.

# 8. Spark SQL View Registration

## Objective

The cleaned datasets from the Silver layer are registered as Spark SQL temporary views. These views provide a unified interface for querying the data and generating business-ready analytical datasets in the Gold layer.

In [57]:
# ==========================================
# Register Silver Tables as SQL Views
# ==========================================

silver_customers.createOrReplaceTempView("customers")

silver_orders.createOrReplaceTempView("orders")

silver_products.createOrReplaceTempView("products")

silver_items.createOrReplaceTempView("order_items")

silver_payments.createOrReplaceTempView("payments")

print("SQL Views created successfully.")

SQL Views created successfully.


## Summary

The Silver datasets have been registered as SQL views. These views will be used to integrate the datasets and generate business-ready analytical tables in the following sections.

# 9. Customer 360

## Objective

Customer information is distributed across multiple operational datasets. This section integrates customer, order, payment, and order item information into a unified Customer 360 view.

The resulting dataset provides a consolidated view of customer purchasing activity and supports customer-centric business analysis.

In [59]:
customer_360 = spark.sql("""

SELECT

c.customer_id,

c.customer_unique_id,

c.customer_city,

c.customer_state,

COUNT(DISTINCT o.order_id) AS total_orders,

ROUND(SUM(p.payment_value),2) AS total_spent,

ROUND(AVG(p.payment_value),2) AS average_order_value,

MIN(o.order_purchase_timestamp) AS first_purchase,

MAX(o.order_purchase_timestamp) AS last_purchase

FROM customers c

LEFT JOIN orders o
ON c.customer_id=o.customer_id

LEFT JOIN payments p
ON o.order_id=p.order_id

GROUP BY

c.customer_id,
c.customer_unique_id,
c.customer_city,
c.customer_state

""")

In [60]:
customer_360.show(10, truncate=False)

+-----------+------------------+--------------+--------------+------------+-----------+-------------------+-------------------+-------------------+
|customer_id|customer_unique_id|customer_city |customer_state|total_orders|total_spent|average_order_value|first_purchase     |last_purchase      |
+-----------+------------------+--------------+--------------+------------+-----------+-------------------+-------------------+-------------------+
|CUST_000001|UNIQ_000001       |salvador      |BA            |1           |604.97     |604.97             |2022-05-29 22:47:00|2022-05-29 22:47:00|
|CUST_000002|UNIQ_000002       |porto alegre  |RS            |2           |2324.41    |1162.21            |2022-01-29 00:06:00|2022-02-21 07:45:00|
|CUST_000003|UNIQ_000003       |recife        |PE            |6           |10851.05   |1808.51            |2021-08-09 00:33:00|2023-04-18 20:47:00|
|CUST_000004|UNIQ_000004       |salvador      |BA            |1           |1864.77    |1864.77            |2021-

## Summary

The Customer 360 dataset provides a unified customer profile by combining customer details, purchasing history, and payment information into a single analytical view.

# 10. Gold Layer

## Objective

The Gold layer contains business-ready datasets generated from the Silver layer. These datasets are designed for reporting, dashboarding, and business decision-making.

Each Gold table addresses a specific business requirement defined in the project.

In [62]:
# ==========================================
# Gold Layer Storage
# ==========================================

GOLD_PATH = "gold"

gold_tables = {
    "customer_360": f"{GOLD_PATH}/customer_360",
    "monthly_revenue": f"{GOLD_PATH}/monthly_revenue",
    "trending_products": f"{GOLD_PATH}/trending_products",
    "customer_segments": f"{GOLD_PATH}/customer_segments",
    "payment_summary": f"{GOLD_PATH}/payment_summary",
    "product_rank": f"{GOLD_PATH}/product_rank"
}

In [63]:
gold_monthly_revenue = spark.sql("""

SELECT

date_format(order_purchase_timestamp,'yyyy-MM') AS month,

ROUND(SUM(payment_value),2) AS total_revenue

FROM orders o

JOIN payments p

ON o.order_id=p.order_id

GROUP BY month

ORDER BY month

""")

gold_monthly_revenue.show()

+-------+-------------+
|  month|total_revenue|
+-------+-------------+
|2021-01|   2517124.73|
|2021-02|   2292457.68|
|2021-03|   2634809.75|
|2021-04|   2621228.92|
|2021-05|   2632810.16|
|2021-06|   2523478.29|
|2021-07|   2622612.11|
|2021-08|   2523118.71|
|2021-09|    2587478.9|
|2021-10|   2646900.28|
|2021-11|    2545607.3|
|2021-12|   2631296.39|
|2022-01|   2657294.01|
|2022-02|    2287956.9|
|2022-03|   2554865.32|
|2022-04|   2591682.12|
|2022-05|    2528428.2|
|2022-06|   2596770.17|
|2022-07|   2677574.85|
|2022-08|   2683212.76|
+-------+-------------+
only showing top 20 rows


In [64]:
gold_trending_products = spark.sql("""

SELECT

oi.product_id,

COUNT(*) AS total_units_sold,

ROUND(SUM(oi.price),2) AS total_sales

FROM order_items oi

GROUP BY oi.product_id

ORDER BY total_units_sold DESC

LIMIT 10

""")

gold_trending_products.show(truncate=False)

+-----------+----------------+-----------+
|product_id |total_units_sold|total_sales|
+-----------+----------------+-----------+
|PROD_001281|51              |57027.12   |
|PROD_000005|51              |66438.26   |
|PROD_000480|50              |58277.57   |
|PROD_001488|47              |60107.4    |
|PROD_001159|47              |61879.35   |
|PROD_001527|46              |55634.56   |
|PROD_001823|45              |51427.92   |
|PROD_002151|45              |54524.1    |
|PROD_000034|44              |52511.76   |
|PROD_001490|44              |62954.22   |
+-----------+----------------+-----------+



In [65]:
gold_customer_segments = spark.sql("""

SELECT

o.customer_id,

ROUND(SUM(p.payment_value),2) AS total_spent,

CASE

WHEN SUM(p.payment_value) >= 1000 THEN 'Premium'

WHEN SUM(p.payment_value) >= 500 THEN 'Gold'

WHEN SUM(p.payment_value) >= 200 THEN 'Silver'

ELSE 'Regular'

END AS customer_segment

FROM orders o

JOIN payments p

ON o.order_id=p.order_id

GROUP BY o.customer_id

ORDER BY total_spent DESC

""")

gold_customer_segments.show()

+-----------+-----------+----------------+
|customer_id|total_spent|customer_segment|
+-----------+-----------+----------------+
|CUST_003170|   22329.52|         Premium|
|CUST_004705|   20375.51|         Premium|
|CUST_004195|   20153.12|         Premium|
|CUST_009641|   20150.59|         Premium|
|CUST_008617|   19552.32|         Premium|
|CUST_002328|   18738.61|         Premium|
|CUST_011464|   18721.58|         Premium|
|CUST_005993|   18466.66|         Premium|
|CUST_002083|   18457.13|         Premium|
|CUST_003253|   18419.99|         Premium|
|CUST_004690|   18259.54|         Premium|
|CUST_002487|    18074.9|         Premium|
|CUST_007247|   17982.25|         Premium|
|CUST_010464|   17835.41|         Premium|
|CUST_011785|   17790.22|         Premium|
|CUST_001675|   17718.42|         Premium|
|CUST_001442|   17712.13|         Premium|
|CUST_013656|   17699.52|         Premium|
|CUST_012609|   17569.23|         Premium|
|CUST_005516|   17513.55|         Premium|
+----------

In [66]:
gold_payment_summary = spark.sql("""

SELECT

payment_type,

COUNT(*) AS total_transactions,

ROUND(SUM(payment_value),2) AS total_payment

FROM payments

GROUP BY payment_type

ORDER BY total_payment DESC

""")

gold_payment_summary.show()

+------------+------------------+-------------+
|payment_type|total_transactions|total_payment|
+------------+------------------+-------------+
| credit_card|             28563|3.775327204E7|
|  debit_card|              9631|1.282442745E7|
|     voucher|              9663|1.281592171E7|
|      boleto|              9531|1.265283389E7|
+------------+------------------+-------------+



In [67]:
gold_product_rank = spark.sql("""

SELECT

product_category_name,

product_id,

total_sales,

RANK() OVER(

PARTITION BY product_category_name

ORDER BY total_sales DESC

) AS product_rank

FROM

(

SELECT

p.product_category_name,

oi.product_id,

COUNT(*) AS total_sales

FROM order_items oi

JOIN products p

ON oi.product_id=p.product_id

GROUP BY

p.product_category_name,

oi.product_id

)

""")

gold_product_rank.show()

+---------------------+-----------+-----------+------------+
|product_category_name| product_id|total_sales|product_rank|
+---------------------+-----------+-----------+------------+
|           automotive|PROD_000406|         44|           1|
|           automotive|PROD_002030|         43|           2|
|           automotive|PROD_002498|         41|           3|
|           automotive|PROD_000831|         41|           3|
|           automotive|PROD_001217|         40|           5|
|           automotive|PROD_001998|         40|           5|
|           automotive|PROD_001166|         40|           5|
|           automotive|PROD_001871|         39|           8|
|           automotive|PROD_001117|         39|           8|
|           automotive|PROD_000867|         38|          10|
|           automotive|PROD_001318|         38|          10|
|           automotive|PROD_002866|         38|          10|
|           automotive|PROD_002460|         37|          13|
|           automotive|P

In [68]:
gold_monthly_revenue.write.mode("overwrite").parquet(
    gold_tables["monthly_revenue"]
)

gold_trending_products.write.mode("overwrite").parquet(
    gold_tables["trending_products"]
)

customer_360.write.mode("overwrite").parquet(
    gold_tables["customer_360"]
)

gold_customer_segments.write.mode("overwrite").parquet(
    gold_tables["customer_segments"]
)

gold_payment_summary.write.mode("overwrite").parquet(
    gold_tables["payment_summary"]
)

gold_product_rank.write.mode("overwrite").parquet(
    gold_tables["product_rank"]
)

print("Gold layer created successfully.")

Gold layer created successfully.


In [69]:
print("=" * 60)
print("Gold Layer Validation")
print("=" * 60)

print(f"Customer 360       : {customer_360.count():>10,}")
print(f"Monthly Revenue    : {gold_monthly_revenue.count():>10,}")
print(f"Trending Products  : {gold_trending_products.count():>10,}")
print(f"Customer Segments  : {gold_customer_segments.count():>10,}")
print(f"Payment Summary    : {gold_payment_summary.count():>10,}")
print(f"Product Ranking    : {gold_product_rank.count():>10,}")

Gold Layer Validation
Customer 360       :     15,000
Monthly Revenue    :         30
Trending Products  :         10
Customer Segments  :     14,481
Payment Summary    :          4
Product Ranking    :      3,000


# 11. Slowly Changing Dimension (SCD Type 2)

## Objective

The Customer Dimension is managed using Slowly Changing Dimension (SCD) Type 2 to preserve historical customer information.

Whenever a customer attribute changes, the existing record is expired by updating its end date and setting it as inactive. A new version of the record is then inserted as the current record.

In [74]:
from pyspark.sql.functions import current_date, lit

customer_dimension = (

    silver_customers

    .withColumn("effective_date", current_date())

    .withColumn("end_date", lit(None).cast("date"))

    .withColumn("is_current", lit(True))

)

customer_dimension.show(5, truncate=False)

+-----------+------------------+------------------------+-------------+--------------+--------------+--------+----------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city|customer_state|effective_date|end_date|is_current|
+-----------+------------------+------------------------+-------------+--------------+--------------+--------+----------+
|CUST_000001|UNIQ_000001       |13278                   |salvador     |BA            |2026-07-19    |NULL    |true      |
|CUST_000002|UNIQ_000002       |42098                   |porto alegre |RS            |2026-07-19    |NULL    |true      |
|CUST_000003|UNIQ_000003       |28289                   |recife       |PE            |2026-07-19    |NULL    |true      |
|CUST_000004|UNIQ_000004       |98696                   |salvador     |BA            |2026-07-19    |NULL    |true      |
|CUST_000005|UNIQ_000005       |21395                   |maceio       |AL            |2026-07-19    |NULL    |true      |
+-----------+-----------

In [75]:
from pyspark.sql.functions import when, col

updated_customers = (

    silver_customers

    .withColumn(

        "customer_city",

        when(
            col("customer_id") == silver_customers.first()["customer_id"],
            "Updated_City"
        ).otherwise(col("customer_city"))

    )

)

In [76]:
changed_records = (

    customer_dimension.alias("old")

    .join(

        updated_customers.alias("new"),

        "customer_id"

    )

    .filter(

        col("old.customer_city") != col("new.customer_city")

    )

)

changed_records.select(
    "customer_id",
    "old.customer_city",
    "new.customer_city"
).show(truncate=False)

+-----------+-------------+-------------+
|customer_id|customer_city|customer_city|
+-----------+-------------+-------------+
|CUST_000001|salvador     |Updated_City |
+-----------+-------------+-------------+



In [78]:
expired_records = (

    customer_dimension.alias("old")

    .join(
        changed_records.select("customer_id"),
        "customer_id"
    )

    .withColumn("end_date", current_date())

    .withColumn("is_current", lit(False))

)

In [79]:
new_records = (

    updated_customers.alias("new")

    .join(
        changed_records.select("customer_id"),
        "customer_id"
    )

    .withColumn("effective_date", current_date())

    .withColumn("end_date", lit(None).cast("date"))

    .withColumn("is_current", lit(True))

)

In [80]:
unchanged_records = (

    customer_dimension.alias("old")

    .join(

        changed_records.select("customer_id"),

        "customer_id",

        "left_anti"

    )

)

In [81]:
customer_dimension_scd2 = (

    unchanged_records

    .unionByName(expired_records)

    .unionByName(new_records)

)

In [82]:
customer_dimension_scd2.orderBy(
    "customer_id",
    "effective_date"
).show(20, truncate=False)

+-----------+------------------+------------------------+--------------+--------------+--------------+----------+----------+
|customer_id|customer_unique_id|customer_zip_code_prefix|customer_city |customer_state|effective_date|end_date  |is_current|
+-----------+------------------+------------------------+--------------+--------------+--------------+----------+----------+
|CUST_000001|UNIQ_000001       |13278                   |Updated_City  |BA            |2026-07-19    |NULL      |true      |
|CUST_000001|UNIQ_000001       |13278                   |salvador      |BA            |2026-07-19    |2026-07-19|false     |
|CUST_000002|UNIQ_000002       |42098                   |porto alegre  |RS            |2026-07-19    |NULL      |true      |
|CUST_000003|UNIQ_000003       |28289                   |recife        |PE            |2026-07-19    |NULL      |true      |
|CUST_000004|UNIQ_000004       |98696                   |salvador      |BA            |2026-07-19    |NULL      |true      |


In [83]:
customer_dimension_scd2.write.mode("overwrite").parquet(
    f"{GOLD_PATH}/customer_dimension_scd2"
)

print("SCD Type 2 table created successfully.")

SCD Type 2 table created successfully.


# 12. Business Insights

The Gold layer provides integrated datasets for business reporting and analysis.

Based on the generated analytical tables:

- Customer 360 provides a consolidated view of customer purchasing activity.
- Monthly revenue analysis identifies the highest revenue period.
- Product sales analysis highlights the best-performing products.
- Customer segmentation classifies customers according to their spending behaviour.
- Payment analysis identifies the most frequently used payment method.

These analytical datasets can be directly used for reporting, dashboard development, and business decision-making.

In [85]:
# ==========================================
# Business Insights
# ==========================================

total_customers = customer_360.count()
total_orders = silver_orders.count()

highest_revenue_month = (
    gold_monthly_revenue
    .orderBy("total_revenue", ascending=False)
    .first()
)

top_product = (
    gold_trending_products
    .orderBy("total_units_sold", ascending=False)
    .first()
)

top_payment = (
    gold_payment_summary
    .orderBy("total_transactions", ascending=False)
    .first()
)

print("=" * 60)
print("Business Insights")
print("=" * 60)

print(f"Total Customers        : {total_customers:,}")
print(f"Total Orders           : {total_orders:,}")

print(
    f"Highest Revenue Month  : {highest_revenue_month['month']} "
    f"(Revenue = {highest_revenue_month['total_revenue']:.2f})"
)

print(
    f"Top Selling Product    : {top_product['product_id']} "
    f"({top_product['total_units_sold']} units sold)"
)

print(
    f"Most Used Payment Type : {top_payment['payment_type']}"
)

Business Insights
Total Customers        : 15,000
Total Orders           : 50,000
Highest Revenue Month  : 2022-08 (Revenue = 2683212.76)
Top Selling Product    : PROD_001281 (51 units sold)
Most Used Payment Type : credit_card


# 13. Conclusion

A centralized analytics platform was developed for RetailMart using PySpark and the Medallion Architecture.

The pipeline ingests raw CSV files into the Bronze layer, performs data cleaning and standardization in the Silver layer, and generates business-ready datasets in the Gold layer.

The solution includes Customer 360 integration, Slowly Changing Dimension (SCD Type 2), Spark SQL analytics, window functions, and customer segmentation. The resulting Gold datasets support reporting, business intelligence, and decision-making.